# NLP Project 2

**ESILV A4 DIA6 — 2026**



**Authors:** Leo WINTER & Alvaro SERERO

## Table of Contents   
1. [Setup & Imports](#setup)
2. [Load Data](#load)
3. [Baseline ML model](#model1)
4. [Model with embeddings](#model2)
5. [Model with pre-trained embeddings](#model3)
6. [USE](#model4)
7. [LLM](#model5)
8. [Comparaison](#comparaison)


<a id="setup"></a>
## 1. Setup & Imports

In [39]:
from pathlib import Path
import datetime
import os

import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras import regularizers
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalMaxPooling1D, GlobalAveragePooling1D, Dense, Dropout,Conv1D
from tensorflow.keras.callbacks import EarlyStopping

import matplotlib.pyplot as plt
import seaborn as sns



# Setting up the Paths
CURRENT_DIR = Path.cwd()
DATA_PATH = CURRENT_DIR.parent / "data"
MODEL_PATH = CURRENT_DIR.parent / "model"
VISU_PATH = CURRENT_DIR.parent / "visualizations" / "notebook3"
LOG_DIR = CURRENT_DIR.parent / "logs" / "projector"
VISU_PATH.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

<a id="load"></a>
## 2. Load Data

In [6]:
# Load data saved in the second notebook
def load_data():
    path_parquet = DATA_PATH / "reviews_modeled.parquet"
    path_csv = DATA_PATH / "reviews_modeled.csv"

    if path_parquet.exists():
        try:
            df = pd.read_parquet(path_parquet)
            return df
        except Exception as e:
            print(e)

    if path_csv.exists():
        try:
            df = pd.read_csv(path_csv)
            return df
        except Exception as e:
            print(e)
    return None


dataset = load_data()
if dataset is None: 
    dataset = pd.DataFrame() #To not have None warnings
display(dataset.head(2))
data_model = dataset.copy()

,note,auteur,avis,assureur,produit,type,date_publication,date_exp,avis_en,avis_cor,avis_cor_en,year_month,tokens_en,tokens,cluster_en,cluster_fr
0,5,brahim--k-131532,"Meilleurs assurances, prix, solutions, écoute,...",Direct Assurance,auto,train,2021-09-06,2021-09-01,"Best insurance, price, solutions, listening, s...",meilleurs assurances prix solutions écoute rap...,best insurance price solutions listening speed...,2021-09,"['good', 'price', 'solution', 'listen', 'speed...","['meilleur', 'prix', 'solution', 'écoute', 'ra...",1,1
1,4,bernard-g-112497,"je suis globalement satisfait , sauf que vous ...",Direct Assurance,auto,train,2021-05-03,2021-05-01,"I am generally satisfied, except that you have...",je suis globalement satisfait sauf que vous av...,i am generally satisfied except that you have ...,2021-05,"['generally', 'satisfied', 'except', 'problem'...","['globalement', 'satisfait', 'sauf', 'problème...",0,2


In [22]:
def label_sentiment(rating):
    if rating <= 2: return 0 
    if rating == 3: return 1 
    return 2                 

data_model['emotion'] = data_model['note'].apply(label_sentiment)
X_en = data_model['tokens_en'].apply(lambda x: "".join(x))
X_fr = data_model['tokens'].apply(lambda x: "".join(x))
y = data_model['note']
y = y - 1 # To have labels from 0 to 4 and not from 1 to 5

X_train_en, X_test_en, y_train_en, y_test_en = train_test_split(X_en, y, test_size=0.2, random_state=42, stratify=y)
X_train_fr, X_test_fr, y_train_fr, y_test_fr = train_test_split(X_fr, y, test_size=0.2, random_state=42, stratify=y)


<a id="model1"></a>
## 3. Baseline ML model

In [ ]:
def model1(X_train, X_test, y_train, y_test):
    tfidf = TfidfVectorizer(ngram_range=(1,2), sublinear_tf=True, max_features=20_000)
    X_train_tfidf = tfidf.fit_transform(X_train)
    X_test_tfidf = tfidf.transform(X_test)

    model_lr = LogisticRegression(max_iter=1000, class_weight='balanced')
    model_lr.fit(X_train_tfidf, y_train)

    y_pred = model_lr.predict(X_test_tfidf)
    accuracy = accuracy_score(y_test, y_pred)
    print(classification_report(y_test, y_pred))
    
    return tfidf,model_lr,accuracy

In [ ]:
print("English model note:")
tfidf_en, model_lr_en,accuracy1_en = model1(X_train_en, X_test_en, y_train_en, y_test_en)
print("\nFrench model note:")
tfidf_fr, model_lr_fr,accuracy1_fr = model1(X_train_fr, X_test_fr, y_train_fr, y_test_fr)

English model note:
              precision    recall  f1-score   support

           1       0.67      0.66      0.66      1453
           2       0.35      0.38      0.36       743
           3       0.28      0.29      0.28       677
           4       0.42      0.38      0.40       977
           5       0.54      0.55      0.54       969

    accuracy                           0.49      4819
   macro avg       0.45      0.45      0.45      4819
weighted avg       0.49      0.49      0.49      4819


French model note:
              precision    recall  f1-score   support

           1       0.67      0.67      0.67      1453
           2       0.36      0.40      0.38       743
           3       0.31      0.29      0.30       677
           4       0.46      0.42      0.44       977
           5       0.56      0.58      0.57       969

    accuracy                           0.50      4819
   macro avg       0.47      0.47      0.47      4819
weighted avg       0.51      0.50    

In [ ]:
def plot_top_coefficients(vectorizer, model, n_words=10):
    feature_names = vectorizer.get_feature_names_out()
    coefs = model.coef_[0]
    
    top_pos = np.argsort(coefs)[-n_words:]
    top_neg = np.argsort(coefs)[:n_words]
    
    print("Important negative worlds:", [feature_names[i] for i in top_neg])
    print("Important positive worlds:", [feature_names[i] for i in top_pos])

print("English note:")
print(f"accuracy = {accuracy1_en:.2f}")
plot_top_coefficients(tfidf_en, model_lr_en)
print("\nFrench note:")
print(f"accuracy = {accuracy1_fr:.2f}")
plot_top_coefficients(tfidf_fr, model_lr_fr)


English note:
accuracy = 0.49
Important negative worlds: ['satisfied', 'thank', 'good', 'price', 'fast', 'moment', 'quick', 'easy', 'hope', 'satisfied service']
Important positive worlds: ['expensive', 'impossible', 'reimburse', 'pay', 'terminate', 'month', 'incompetent', 'increase', 'avoid', 'flee']

French note:
accuracy = 0.50
Important negative worlds: ['satisfait', 'satisfaite', 'rapide', 'merci', 'prix', 'très', 'espère', 'écoute', 'bon', 'efficace']
Important positive worlds: ['argent', 'incompétent', 'nul', 'aucun', 'déconseille', 'aucune', 'éviter', 'fuyez', 'mois', 'fuir']

English emotion:
accuracy = 0.75
Important negative worlds: ['satisfied', 'thank', 'good', 'fast', 'quick', 'effective', 'easy', 'responsive', 'well', 'practical']
Important positive worlds: ['pay', 'refuse', 'increase', 'incompetent', 'impossible', 'deplorable', 'avoid', 'month', 'terminate', 'flee']

French emotion:
accuracy = 0.78
Important negative worlds: ['satisfait', 'rapide', 'satisfaite', 'merci',

<a id="model2"></a>
## 4. Model with embeddings

In [23]:
max_words = 10000  
max_length = 100     

tokenizer_en = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer_en.fit_on_texts(X_train_en)

X_train_seq_en = tokenizer_en.texts_to_sequences(X_train_en)
X_test_seq_en = tokenizer_en.texts_to_sequences(X_test_en)
X_train_pad_en = pad_sequences(X_train_seq_en, maxlen=max_length, padding='post')
X_test_pad_en = pad_sequences(X_test_seq_en, maxlen=max_length, padding='post')


tokenizer_fr = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer_fr.fit_on_texts(X_train_fr)

X_train_seq_fr = tokenizer_fr.texts_to_sequences(X_train_fr)
X_test_seq_fr = tokenizer_fr.texts_to_sequences(X_test_fr)
X_train_pad_fr = pad_sequences(X_train_seq_fr, maxlen=max_length, padding='post')
X_test_pad_fr = pad_sequences(X_test_seq_fr, maxlen=max_length, padding='post')

In [ ]:
def model2(max_words,embedding_dim = 64):
   
    model = Sequential([
        Embedding(input_dim=max_words, output_dim=embedding_dim),

        Conv1D(64, 5, activation='relu', padding='same'),
        GlobalMaxPooling1D(),
        
        Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001)),

        Dropout(0.5),
        Dense(5, activation='softmax')
    ])

    model.compile(loss='sparse_categorical_crossentropy', 
                      optimizer='adam', 
                      metrics=['accuracy'])
    return model

model_embeddings_en = model2(max_words=max_words)
model_embeddings_en.build(input_shape=(None, max_length))
model_embeddings_en.summary()

model_embeddings_fr = model2(max_words=max_words)

Model: "sequential_23"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_23 (Embedding)        │ (None, 100, 64)        │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_10 (Conv1D)              │ (None, 100, 64)        │        20,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_8          │ (None, 64)             │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_29 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_52 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_53 (Dense)                │ (None, 5)              │           165 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 662,789 (2.53 MB)

 Trainable params: 662,789 (2.53 MB)

 Non-trainable params: 0 (0.00 B)

In [54]:
log_dir = str(LOG_DIR / "fit" / f"{datetime.datetime.now().strftime("%Y%m%d-%H%M%S")}")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stop = EarlyStopping(
    monitor='val_loss', 
    patience=3,          
    restore_best_weights=True 
)

print("English model :")
history = model_embeddings_en.fit(
    X_train_pad_en, y_train_en,
    epochs=15,
    validation_data=(X_test_pad_en, y_test_en),
    callbacks=[tensorboard_callback,early_stop],
    batch_size=32
)

print("\nFrench model :")
history = model_embeddings_fr.fit(
    X_train_pad_fr, y_train_fr,
    epochs=15,
    validation_data=(X_test_pad_fr, y_test_fr),
    callbacks=[tensorboard_callback,early_stop],
    batch_size=32
)

English model :
Epoch 1/15
603/603 ━━━━━━━━━━━━━━━━━━━━ 10s 13ms/step - accuracy: 0.4442 - loss: 1.2818 - val_accuracy: 0.4924 - val_loss: 1.1374
Epoch 2/15
603/603 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.5010 - loss: 1.1138 - val_accuracy: 0.5001 - val_loss: 1.1266
Epoch 3/15
603/603 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.5311 - loss: 1.0515 - val_accuracy: 0.4858 - val_loss: 1.1297
Epoch 4/15
603/603 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.5637 - loss: 0.9874 - val_accuracy: 0.4974 - val_loss: 1.1464
Epoch 5/15
603/603 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.6096 - loss: 0.9137 - val_accuracy: 0.5028 - val_loss: 1.1926

French model :
Epoch 1/15
 82/603 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.3010 - loss: 1.6193

KeyboardInterrupt: 

In [52]:
loss_en, accuracy2_en = model_embeddings_en.evaluate(X_test_pad_en, y_test_en,verbose=0)
loss_fr, accuracy2_fr = model_embeddings_fr.evaluate(X_test_pad_fr, y_test_fr,verbose=0)

print("English model :")
print(f"Loss     : {loss_en:.4f}")
print(f"Accuracy : {accuracy2_en:.2}")
print("\nFrench model :")
print(f"Loss     : {loss_fr:.4f}")
print(f"Accuracy : {accuracy2_fr:.2}")

English model :
Loss     : 1.1088
Accuracy : 0.51

French model :
Loss     : 1.0818
Accuracy : 0.53


<a id="model3"></a>
## 5. Model with pre-trained embeddings

<a id="model4"></a>
## 6. USE

<a id="model5"></a>
## 7. LLM

<a id="comparaison"></a>
## 8. Comparaison